# MediVision AI — `fracture_det` (full profile)

Auto-generated by `ml/kaggle/generate.py`. Trains OUR model from scratch / generic ImageNet init on free public data. Estimated GPU time: ~1.5 h (T4/P100).

**Research & educational prototype — not a medical device.**

In [ ]:
import base64, io, os, subprocess, sys, tarfile, pathlib, shutil
pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm",
                      "segmentation-models-pytorch", "medmnist", "tokenizers", "onnx", "onnxruntime",
                      "onnxscript", "pydicom", "rapidfuzz"], check=False)
import socket, torch
try:
    socket.gethostbyname("pypi.org"); INTERNET = True
except OSError:
    INTERNET = False
print("internet:", INTERNET, "| GPU:", torch.cuda.is_available())
if not INTERNET:
    print("WARNING: no internet in this kernel. Verify your phone number on kaggle.com "
          "(Settings -> Phone verification) to enable internet + GPU; preinstalled packages are used.")
    os.environ["PRETRAINED_BACKBONE"] = "false"  # ImageNet weights need internet -> train from scratch
    print("PRETRAINED_BACKBONE=false (no internet to fetch generic ImageNet weights)")
WORK = pathlib.Path("/kaggle/working/medivision"); WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK); sys.path.insert(0, str(WORK))
BUNDLE = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE)), mode="r:gz").extractall(WORK)
os.environ["MEDIVISION_DATA_DIR"] = str(WORK / "data")
os.environ["PYTHONPATH"] = str(WORK)
print("bundle extracted:", sorted(os.listdir(WORK)))

In [ ]:
LINKS = {"fracatlas": "fracatlas"}
for key, slug in LINKS.items():
    src = pathlib.Path("/kaggle/input") / slug
    if not src.exists():  # newer Kaggle layouts nest mounts, e.g. /kaggle/input/datasets/<owner>/<slug>
        hits = [p for depth in ("*", "*/*", "*/*/*") for p in pathlib.Path("/kaggle/input").glob(depth)
                if p.is_dir() and p.name == slug]
        src = hits[0] if hits else src
    d = WORK / "data" / "raw" / key; d.mkdir(parents=True, exist_ok=True)
    if not (d / "src").exists() and src.exists():
        os.symlink(src, d / "src")
    print(key, "->", src, src.exists())
print("inputs:", [str(p) for p in pathlib.Path("/kaggle/input").glob("*/*")][:20])
for k, v in {}.items():
    os.environ[k] = v

In [ ]:
from ml.data import download, prepare
for key in []:
    print(key, download.fetch(download.BY_KEY[key]))
for spec in []:
    import medmnist
    flag, size = spec.split(":")
    cls = getattr(medmnist, medmnist.INFO[flag]["python_class"])
    for split in ("train", "val", "test"):
        cls(split=split, download=True, root=str(WORK / "data" / "raw" / flag), size=int(size))
from ml.data.prepare import *  # noqa
prepare_fracatlas(768)

In [ ]:
PROFILE = "full"
for m in ["fracture_det"]:
    subprocess.run([sys.executable, "-m", "ml.train", "--model", m, "--profile", PROFILE], check=True)

In [ ]:
for m in ["fracture_det"]:
    subprocess.run([sys.executable, "-m", "ml.evaluate", "--model", m, "--profile", PROFILE], check=False)
for m in ["fracture_det"]:
    subprocess.run([sys.executable, "-m", "ml.export_onnx", "--model", m, "--profile", PROFILE], check=False)

In [ ]:
OUT = pathlib.Path("/kaggle/working/output"); OUT.mkdir(exist_ok=True)
for m in ["fracture_det"]:
    src = WORK / "artifacts" / m / PROFILE
    if src.exists():
        shutil.copytree(src, OUT / m / PROFILE, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("checkpoints"))
shutil.rmtree(WORK / "data", ignore_errors=True)  # keep the kernel output small
print(sorted(str(p.relative_to(OUT)) for p in OUT.rglob("*") if p.is_file()))